# Outline

The purpose of this script is to derrive post-hoc measures from the mode timecourses and coherence matrices. 
We'll epoch the mode timecourses (relative to correct nogo stimulus onsets and button presses) and compute mean amplitude over the first 750 ms.
We'll take the top 3% of connectivity edges _at the group level_ and compute participant means across those edges

# Prepare

In [ ]:
# Imports
import mne
import os
import re
import pandas as pd
import numpy as np
import glob
import datetime
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
from nilearn import image, datasets
from plotting_functions import *  # custom functions for plotting modes
import seaborn as sns
from scipy.stats import gaussian_kde, zscore
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr, linregress

# Imports for R functionality
from rpy2.robjects.packages import importr
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter

mgcv = importr('mgcv')



In [ ]:
# Write out data?
writeOK=False

In [ ]:
# Suppress mne output 
mne.set_log_level('ERROR')

In [ ]:
# Read config file and define paths
now = datetime.datetime.now()
datetime_str = now.strftime("%Y-%m-%d_%H%M")

# Read config file and define paths
with open('0_config.yaml', 'r') as file:
    config = yaml.safe_load(file)
    
subjects_dir = config['dirs']['recon_dir']
proc_dir = config['dirs']['proc_dir']
atlas_dir = config['dirs']['atlas_dir']
model_dir = config['dirs']['model_dir']
results_dir = config['dirs']['results_dir']

# Update model dir for the appropriate training run (the one with the lowest variational free energy)
training_run = config['analysis_params']['selected_training_run']
model_dir = f'{model_dir}_run{training_run}'

# Define directory containing posthoc output
posthoc_dir = os.path.join(model_dir, 'posthoc')
os.makedirs(posthoc_dir, exist_ok=True)


In [ ]:
# Plotting params
colours = config['misc']['colours']
n_modes = 6

# Other params
exclude = config['misc']['exclusion_col']
groups = config['misc']['groups']
orig_mode_order = config['posthoc']['orig_mode_order']
new_mode_order = config['posthoc']['new_mode_order']
window = (-0.2, 1.0)  # time window for epoching



In [ ]:
# Define a helper function to pull the most recent version of a file
# Note that this sleects file based on the date & time that they were last
# modified, not the actual datetime printed in the filename 
def recent_fname(path):
    
    # Takes a partial file path
    files = glob.glob(path)
    most_recent_file = max(files, key=os.path.getmtime)
    
    return most_recent_file

# Define function for classifying nogo events as correct/incorrect
def split_events_by_accuracy(events, nogo_codes, go_codes, button_codes):

    # Filter out None values from codes
    nogo_codes = [code for code in nogo_codes if code is not None]
    go_codes = [code for code in go_codes if code is not None]

    # Filter events to only include Go, Nogo, and Button events
    valid_codes = nogo_codes + go_codes + button_codes
    events = np.array([event for event in events if event[2] in valid_codes])
    
    correct_nogo_events = []
    incorrect_nogo_events = []
    correct_go_events = []
    incorrect_go_events = []
 
    for i, event in enumerate(events):
        if event[2] in nogo_codes or event[2] in go_codes:
            # find next trial event (Go or Nogo)
            next_trial_idx = None
            for j in range(i+1, len(events)):
                if events[j][2] in (nogo_codes + go_codes):
                    next_trial_idx = j
                    break
            # define search window (events between current trial and next trial, or until end)
            start = i + 1
            end = next_trial_idx if next_trial_idx is not None else len(events)
            
            # check if any button press occurred in that window
            has_button = any(events[k][2] in button_codes for k in range(start, end))
            
            if event[2] in nogo_codes:
                if has_button:
                    incorrect_nogo_events.append(event)
                else:
                    correct_nogo_events.append(event)
            elif event[2] in go_codes:
                if has_button:
                    correct_go_events.append(event)
                else:
                    incorrect_go_events.append(event)

    # # Verification: Compute % correct for each trial type
    # n_total_go = len(correct_go_events) + len(incorrect_go_events)
    # n_total_nogo = len(correct_nogo_events) + len(incorrect_nogo_events)
    
    # if n_total_go > 0:
    #     go_accuracy = len(correct_go_events) / n_total_go * 100
    #     print(f'%Go correct: {go_accuracy:.2f}%')
    
    # if n_total_nogo > 0:
    #     nogo_accuracy = len(correct_nogo_events) / n_total_nogo * 100
    #     print(f'%Nogo correct: {nogo_accuracy:.2f}%')
    
    return correct_nogo_events, incorrect_nogo_events, correct_go_events

In [ ]:
# Files to load
subjects_fname = recent_fname(os.path.join(results_dir, 'subject_order_*.txt')) 
atlas_fname = os.path.join(atlas_dir, 'MEG_atlas_38_regions_4D.nii.gz')
master_fname = config['filenames']['masterlist_fname'] 

# Load list of subjects 
subjects = np.loadtxt(subjects_fname, dtype=str).tolist()

# Read masterlist and drop excluded subjects
masterlist = masterlist[(masterlist[exclude] == 0) & (masterlist['Dx'].isin(groups))]

In [ ]:
# Load DyNeMo outputs
alpha = np.load(os.path.join(model_dir, 'inf_params', 'alp_rw.pkl'),allow_pickle=True)
covs = np.load(os.path.join(model_dir, 'inf_params', 'covs.npy'))
psd = np.load(os.path.join(model_dir, 'spectra', 'psd_rs.npy'))
psd_base = psd[:,1]
psd_coef = psd[:,0]
coh = np.load(os.path.join(model_dir, 'spectra', 'coh.npy'))
freqs = np.load(os.path.join(model_dir, 'spectra', 'f.npy'))

# Mode timecourse / activation analyses

In [ ]:
# Define empty lists to store posthoc output...

# Demographics
all_ages = []               
all_sexes = []              
all_dxs = []   
all_uniqueIDs = []             

# Epoch data for Correct Nogo events (CNg) and Button Press events (BP)
all_CNg_data = []
all_BP_data = []

# We'll also get happy and angry data seperately
all_happy_data = []
all_angry_data = []

# Mean CNg amplitude over the first 750 ms
all_CNg_amp = []

# A long-format list of subjectIDs, which will help us store out single-epoch data
all_subjects_expand = []

# Loop through subjects. For each subject, we'll get...
# 1. Epoched mode timecourses (separately for 
for s, subject in enumerate(subjects):
       
    # Pull demographic information for this subject from the masterlist
    row =  masterlist.loc[masterlist['SubjectID'] == subject].squeeze()
    age, sex, dx, uniqueID = row[['Age', 'Sex', 'Dx', 'UniqueID']]
    
    # Load ortho data for this subject
    raw_fname = recent_fname(os.path.join(proc_dir, subject, f'*_{subject}_source_orth-raw.fif'))
    raw = mne.io.Raw(raw_fname, preload=True)
    
    # Define alpha and bad samples for this subject
    sub_alpha = alpha[s].T
    bads = np.isnan(raw.get_data(reject_by_annotation='nan')[0,:])
    
    # Pad alpha
    keep_index = np.ones(len(raw))
    keep_index[bads] = 0
    n_embeddings = 15
    embed_loss = (n_embeddings-1) // 2
    keep_index[:embed_loss] = 0
    keep_index[-embed_loss:] = 0
    keep_pos = np.squeeze(np.argwhere(keep_index==1))
    alpha_pad = np.zeros((sub_alpha.shape[0], len(raw)))
    alpha_pad[:,keep_pos[:sub_alpha.shape[1]]] = sub_alpha.copy() 
    
    # Make mne.Raw object from alpha
    alpha_info = mne.create_info(['mode' + str(mode+1) for mode in range(n_modes)], 250)
    alpha_raw = mne.io.RawArray(alpha_pad, alpha_info)
    alpha_raw.set_meas_date(raw.info['meas_date'])
    alpha_raw.set_annotations(raw.annotations)
    
    # Get events
    events, ids = mne.events_from_annotations(alpha_raw)  
    
    # Get event codes for Go and Nogo trials. Note that some participants have slightly 
    # different event coding, with 'Target/NonTarget' instead of 'Go/Nogo', etc
    nogo_codes = [ids.get('Nogo'), ids.get('NonTarget')]
    go_codes = [ids.get('Go'), ids.get('Target')]
    
    # also get codes for Happy and Angry
    happy_code = ids.get('Happy')
    angry_code = ids.get('Angry')
    emotion_dict = {'Happy': happy_code, 'Angry': angry_code}
    emotion_codes = [happy_code, angry_code]
    angry_code = ids.get('Angry')

    # For now, skip any subjects without happy/angry codes
    if pd.isna(happy_code) or pd.isna(angry_code):
        print(f'{subject} has no happy or angry codes')
        continue

    # Compute the proportion of trials where a Happy face was presented
    all_trials = events[np.isin(events[:,2],[go_codes,nogo_codes])]  # includes ALL go and nogo events
    proportion_happy = len(events[events[:,2]==happy_code]) / len(all_trials)
    
    # Raise an error if subjects havew 100% happy or 100% angry
    assert proportion_happy != 1.0, f'Subject {subject} has 100% happy trials'
    assert proportion_happy != 0.0, f'Subject {subject} has 100% angry trials'    
    
    # Get event codes for button presses. As above there is some variation here.
    # We'll look for any events containing the substring 'Button' or 'Correct'. This means 
    # that in some cases we're getting both correct and incorrect trials
    pattern = re.compile(r'(Button|Correct)', re.IGNORECASE)
    button_ids = {k: v for k, v in ids.items() if pattern.search(k)}
    button_codes = list(button_ids.values())
    
    # Get correct Nogo events. Since Correct and Incorrect were not consistently 
    # coded in the experiment, we'll determine accuracy based on button press events. 
    # A correct Nogo trial is one that is not proceeded by a button press before the 
    # next trial...    
    correct_nogo_events, _, correct_go_events = split_events_by_accuracy(events, nogo_codes, 
                                                        go_codes, button_codes)

    correct_nogo_events = np.array(correct_nogo_events)
           
    
    # Happy/Angry triggers occur simultaneously with nogo triggers. So, let's pull out only 
    # correct emotion trials, based on correct nogo timings
    correct_nogo_onsets = set(correct_nogo_events[:, 0])  # times for correct nogo events
    correct_emotion_nogo_events = np.array([e for e in events if np.isin(e[2], emotion_codes) and e[0] in correct_nogo_onsets])

    
    # Ensure that the event timings for emotional events exactly match those of the original correct events
    assert correct_nogo_events[:,0].all() == correct_emotion_nogo_events[:,0].all(), "ERROR: Event onset mismatch between emotional events and correct events"
        
    # Epoch correct nogo trials, seperately for happy and angry
    emotional_faces_epochs = mne.Epochs(alpha_raw,  
                        correct_emotion_nogo_events,   # we'll only take correct nogo trials
                        event_id=emotion_dict, 
                        event_repeated='error',  
                        tmin=window[0], tmax=window[1], baseline=None, preload=True)

    
    # Get data for happy and angry trials, and also for ALL correct nogo trials collapsed across emotion
    CNg_happy_data = emotional_faces_epochs["Happy"].get_data()
    CNg_angry_data = emotional_faces_epochs["Angry"].get_data()
    CNg_data = emotional_faces_epochs.get_data()

    # Compute CNG amplitude over the 0.75 seconds (0.75 is the shortest ISI of any participant)
    CNg_amp = np.mean(emotional_faces_epochs.copy().crop(0, 0.75).get_data(), axis=2)  # shape (n_epochs, n_modes)

    # # Alternatively we could epoch based on correct nogo events, without splitting by emotion. Results are the same.
    # CNg_epochs = mne.Epochs(alpha_raw, 
    #                     correct_nogo_events, 
    #                     tmin=-0.2, tmax=1, baseline=None, preload=True)
    # emotional_faces_epochs.get_data().all() == CNg_epochs.get_data().all()
    

    # Also get button press epochs (BP_epochs). Since we don't need to classify these
    # by accuracy we can just grab all button press events from the original events array
    BP_epochs = mne.Epochs(alpha_raw,  
                        events, event_id=button_codes, 
                        event_repeated='merge',  # some participants have multiple coincident events for button presses (buttonRight, buttonCorrect, etc...)
                        tmin=window[0], tmax=window[1], baseline=None, preload=True)
    BP_data = BP_epochs.get_data()

    # Append data to our group lists
    all_ages.append(age)
    all_sexes.append(sex)
    all_dxs.append(dx)
    all_uniqueIDs.append(uniqueID)
    
    all_CNg_data.append(CNg_data)
    all_CNg_amp.append(CNg_amp)
    all_BP_data.append(BP_data)
    
    all_happy_data.append(CNg_happy_data)
    all_angry_data.append(CNg_angry_data)

    # all_subjects_expand is a long-format list of subject IDs with length equal to the number of epochs
    all_subjects_expand.append(np.repeat(subject, CNg_amp.shape[0]))

In [ ]:
# Get mean mode timecourses for correct Nogo trials
times = emotional_faces_epochs.times
baseline = (times>-0.2) * (times < -0.1)
mean_timecourses = np.array([np.mean(inst,0) for inst in all_CNg_data])

# and happy/angry trials
mean_happy_timecourses = np.array([np.mean(inst,0) for inst in all_happy_data])
mean_angry_timecourses = np.array([np.mean(inst,0) for inst in all_angry_data])

# and BP trials
bp_times = BP_epochs.times
bp_baseline = (bp_times>-0.2) * (bp_times < -0.1)
mean_bp_timecourses = np.array([np.mean(inst,0) for inst in all_BP_data])

# Save CNg timecourses and times to disk
if writeOK:
    np.save(os.path.join(posthoc_dir, f'{datetime_str}_CNg_timecourses'), mean_timecourses)
    np.save(os.path.join(posthoc_dir, f'{datetime_str}_CNg_happy_timecourses'), mean_happy_timecourses)
    np.save(os.path.join(posthoc_dir, f'{datetime_str}_CNg_angry_timecourses'), mean_angry_timecourses)
    np.save(os.path.join(posthoc_dir, f'{datetime_str}_CNg_times.csv'), times)

# Mode coherence analysis

In [ ]:
# Define threshold for top% coherence
coh_thresh = 3
top_percentile = 100-coh_thresh

In [ ]:
# Define an empty list to store single-subject coherences for all modes
subject_cohs_all_modes = []

# Loop through modes
for mode in range(n_modes):
    
    # Define empty list to store single-subject coherences, for this mode
    subject_cohs = []

    # Define a frequency range    
    #freq_lims = freq_ranges[orig_mode_order[mode]]
    freq_range = (freqs>=4) * (freqs<=30)
    
    
    # Average over subjects and frequencies. 
    mode_coh = np.mean(coh[:,mode,:,:,:], 0)
    conn = np.nanmean(mode_coh[:,:,freq_range], -1)
    triu = np.triu_indices(mode_coh.shape[0], k=1)
    np.fill_diagonal(conn, np.nan)
    threshold = np.nanpercentile(np.abs(conn[triu]), top_percentile)
    #fig = glass_brain_plot(conn, coords, threshold, 'Mode Coherence')
    
    # Make binary mask of top (3%) connections
    top_mask = np.abs(conn) >= threshold
    top_mask = np.triu(top_mask, k=1) | np.triu(top_mask, k=1).T
    
    # Upper-triangle-only mask for unique edges
    mask_ut = np.zeros_like(top_mask, dtype=bool)
    mask_ut[triu] = True
    top_mask_ut = top_mask & mask_ut
    
    # Get array of coherences (averaged over frequency range) in individual subjects
    subjects_coh = np.nanmean(
        coh[:, mode, :, :, :][:, :, :, freq_range],
        axis=-1
    )
    
    # Mask subject coherence array (only keep top edges from above)
    #subjects_coh_masked = np.zeros_like(subjects_coh)
    subjects_coh_masked = np.full_like(subjects_coh, np.nan)
    subjects_coh_masked[:, top_mask_ut] = subjects_coh[:, top_mask_ut]
        
    # Extract the masked coherence for each subject
    for s, subject in enumerate(subjects):
        
        # Get top edges for this subject
        subj_coh = subjects_coh_masked[s]
        
        # Compute the mean
        subject_coh_av = np.nanmean(subj_coh)
        
        # Store out
        subject_cohs.append(subject_coh_av)
        
    # Append the list for this mode to the larger 'all_modes' list
    subject_cohs_all_modes.append(subject_cohs)

# Coerce posthoc output to dataframes

In [ ]:
# Create demographics df
df_demo = pd.DataFrame({
    'SubjectID': subjects,
    'UniqueID': all_uniqueIDs, 
    'Age': all_ages,
    'Sex': all_sexes,
    'Dx': all_dxs   
    })

# Sex is sometimes coded as 'F   '. Fix this. 
df_demo['Sex'] = df_demo['Sex'].str.strip()

# Create a df for CNg amplitude data. 
all_CNg_amp_arr = np.concatenate(all_CNg_amp)
all_subjects_expand_arr = np.concatenate(all_subjects_expand)

df_CNg_amp = pd.DataFrame({
    'SubjectID': all_subjects_expand_arr,
    **{f'mode{i+1}': all_CNg_amp_arr[:, i] for i in range(all_CNg_amp_arr.shape[1])}

    })

# Create a df for coherence data.
subject_cohs_arr = np.asarray(subject_cohs_all_modes)


df_coh = pd.DataFrame({
        'SubjectID': subjects,
        **{f'mode{i+1}_coh': subject_cohs_arr[i,:] for i in range(subject_cohs_arr.shape[0])}  
    
        }) # note that subjects_coh_arr is transposed relative to the amplitude data, hence the switching dimensions

# Finally merge each posthoc dataframe with the demographics df
df_CNg = pd.merge(df_demo, df_CNg_amp, on='SubjectID')
df_coh = pd.merge(df_demo, df_coh, on='SubjectID')


In [ ]:
# Save posthoc output to disk
if writeOK:
    
    # CNg amplitude
    out_fname_CNg = os.path.join(posthoc_dir, f'{datetime_str}_CNg_epoch_amplitudes_750ms.csv')
    df_CNg.to_csv(out_fname_CNg, index=False)
    print(f'Wrote {out_fname_CNg}')
    
    # Coherence
    out_fname_coh = os.path.join(posthoc_dir, f'{datetime_str}_CNg_coh.csv')
    df_coh.to_csv(out_fname_coh, index=False)
    print(f'Wrote {out_fname_coh}')    
    
    # Also write out ages, for convenience
    age_out_fname = os.path.join(posthoc_dir, f'{datetime_str}_all_subject_ages')
    np.save(age_out_fname, all_ages)
    print(f'Wrote {age_out_fname}')    

In [ ]:
# # Quality assurance: If you want to check demographic info for any one subject, use...
# ss = subjects[0]
# df_CNg.loc[df_CNg['SubjectID']==ss]

# Plot mean mode timecourses relative to stimulus onset or button presses

In [ ]:
# Get mean mode timecourses for correct Nogo trials
times = emotional_faces_epochs.times
baseline = (times>-0.2) * (times < -0.1)
mean_timecourses = np.array([np.mean(inst,0) for inst in all_CNg_data])

# and happy/angry trials
mean_happy_timecourses = np.array([np.mean(inst,0) for inst in all_happy_data])
mean_angry_timecourses = np.array([np.mean(inst,0) for inst in all_angry_data])

# and BP trials
bp_times = BP_epochs.times
bp_baseline = (bp_times>-0.2) * (bp_times < -0.1)
mean_bp_timecourses = np.array([np.mean(inst,0) for inst in all_BP_data])

# Save CNg timecourses and times to disk
if writeOK:
    np.save(os.path.join(posthoc_dir, f'{datetime_str}_CNg_timecourses'), mean_timecourses)
    np.save(os.path.join(posthoc_dir, f'{datetime_str}_CNg_happy_timecourses'), mean_happy_timecourses)
    np.save(os.path.join(posthoc_dir, f'{datetime_str}_CNg_angry_timecourses'), mean_angry_timecourses)
    np.save(os.path.join(posthoc_dir, f'{datetime_str}_CNg_times.csv'), times)
    

In [ ]:
# Plot mean timecourses relative to face onset, for all CNg trials
fig, ax = plt.subplots(figsize=(6,3.5))
for mode in range(n_modes):
    mode_timecourse = mean_timecourses[:,mode,:]
    values = np.mean(mode_timecourse, 0)
    values -= np.mean(values[baseline])
    err = np.std(mode_timecourse, 0)/ np.sqrt(mode_timecourse.shape[0])
    line, = ax.plot(times, values, label=orig_mode_order[mode], color=colours[mode])
    colours.append(line.get_color())  # store line colours for later plots
    ax.fill_between(times, values-err, values+err, alpha=0.2, color=colours[mode])
ax.axvline(0, linestyle='--', color=[0.2,0.2,0.2, 0.5], label=None)
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_xticks([-0.2, 0, 0.25, 0.5, 0.75, 1.0])
ax.set_ylabel('Mode Activation', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

# Arrange legend in logical order
handles, labels = ax.get_legend_handles_labels()

order = [labels.index(name) for name in new_mode_order]
handles_ord = [handles[i] for i in order]
labels_ord  = [labels[i] for i in order]

ax.legend(
    handles_ord,
    labels_ord,
    ncol=1,
    fontsize=10,
    frameon=False,
    loc='center',
    bbox_to_anchor=(0.8, 0.8, 0.1, 0.1)
    )

plt.tight_layout()
plt.show()

# Save figure
fig.savefig(os.path.join(model_dir, 'figures', 'CNg_mean_timecourses.png'))


In [ ]:
# Plot happy and angry timecourses relative to face onset, for each mode

for mode in range(n_modes):
    fig, ax = plt.subplots(figsize=(6,3.5))
    for emotion in ['happy', 'angry']:
        
        if emotion=='happy':
            linestyle = '-'
        elif emotion=='angry':
            linestyle='--'
        
        mode_timecourse = eval(f'mean_{emotion}_timecourses[:,mode,:]')
        
        values = np.mean(mode_timecourse, 0)
        values -= np.mean(values[baseline])
        
        
        err = np.std(mode_timecourse, 0)/ np.sqrt(mode_timecourse.shape[0])
        line, = ax.plot(times, values, label=emotion, color=colours[mode], linestyle=linestyle)
        colours.append(line.get_color())  # store line colours for later plots
        ax.fill_between(times, values-err, values+err, alpha=0.2, color=colours[mode])
            
    ax.axvline(0, linestyle='--', color=[0.2,0.2,0.2, 0.5], label=None)
    ax.legend()
    plt.title(orig_mode_order[mode])
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_xticks([-0.2, 0, 0.25, 0.5, 0.75, 1.0])
    ax.set_ylabel('Mode Activation', fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    plt.show()

In [ ]:
# Plot button-press timecourses two sensorimotor modes, relative to button press events
sens_modes = [4,5]

fig, ax = plt.subplots(figsize=(10,6))


for mode in sens_modes:
    mode_timecourse = mean_bp_timecourses[:,mode,:]
    values = np.mean(mode_timecourse, 0)
    values -= np.mean(values[bp_baseline])
    err = np.std(mode_timecourse, 0)/ np.sqrt(mode_timecourse.shape[0])
    line, = ax.plot(times, values, label=orig_mode_order[mode], color=colours[mode])
    colours.append(line.get_color())  # store line colours for later plots
    ax.fill_between(times, values-err, values+err, alpha=0.2, color=colours[mode])
    ax.axvline(0, linestyle='--', color=[0.2,0.2,0.2, 0.5], label=None)
    ax.legend(ncol=1, fontsize=8, frameon=False, loc = 'center left', bbox_to_anchor=(0.7, 0.2, 0.1, 0.1))
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_xticks([-0.2, 0, 0.25, 0.5, 0.75, 1.0])
    ax.set_ylabel('Mode Activation', fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)

# Arrange legend in logical order
handles, labels = ax.get_legend_handles_labels()

plt.tight_layout()
plt.show()

# Save figure
fig.savefig(os.path.join(model_dir, 'figures', 'BP_mean_timecourses.png'))

# Supplemental analysis
Use NHST thin-plate spline regression to model developmental effects on mode activation and coherence.